# Two hours, three layers, one portfolio artifact. End-to-end ownership of one analytics domain.

You own this domain end to end. The analytics team wants to ask "show me revenue by product category over the last 30 days" and your job is to build the data path that returns the right number. Raw clickstream lands messy. You make bronze tolerant, silver clean, gold queryable. Every layer is a design call you defend in the reflection. By the end of the lab you have a markdown system-design writeup the interview panel will read and ask you to walk through.

The six deliverables map to six checkpoints:
1. Raw seed exists; bronze stands up tolerating malformed records (parse errors get tagged, not dropped).
2. Silver is typed, deduped, late-arrival tolerant, partitioned by event_date.
3. Gold star schema: one fact table, three conformed dimensions; foreign keys consistent.
4. The analyst BI query returns the expected number against gold; ground-truth from silver matches exactly.
5. Iceberg snapshot history on gold shows the expected number of commits (architectural cleanliness).
6. Portfolio artifact: `portfolio/system-design.md` exists with required sections; 800-1500 words; preserved past cleanup.

**Time.** About 120 minutes hands-on. Three Glue ETL runs at the 10-minute minimum is most of the wall clock. A built-in mid-point (after Checkpoint 3) is a natural pause if you need to come back.

**Cost.** This lab costs about $2.50 on a clean run. The most expensive lab in the catalog. Glue ETL is the line item: bronze + silver + gold + reruns lands around six 2-DPU runs at 4-6 minutes each. Athena scans for the BI query + ground-truth + validators: ~$0.50. Iceberg storage + metadata: free. Realistic upper bound with debug reruns: $5.00.

**No critical resources at steady state.** Glue ETL only bills while a job is running.

**Multi-session.** 24-hour JWT. The natural resume point is after Checkpoint 3 (silver complete).

**Portfolio mode.** The cleanup cell preserves `portfolio/system-design.md`. The companion panel surfaces a download link after the lab completes.

This lab is part of the Labrun Data Engineering job-prep track, Warehouse & Lakehouse Mastery sub-track. It is a killer lab.


In [ ]:
!pip install --quiet labrun-checks==0.7.0 boto3


In [ ]:
# Imports and constants.

import atexit
import io
import json
import re
import sys
import time
import uuid
import zipfile
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import CheckpointResult, CleanupResource, check, cleanup, register, run_cleanup

LAB_SLUG = "build-the-medallion"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG
REGION = "us-east-1"

BUCKET_NAME = None
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
BRONZE_TABLE = "clickstream_bronze"
SILVER_TABLE = "clickstream_silver"
FACT_TABLE = "fact_clickstream"
DIM_PRODUCT = "dim_product"
DIM_CUSTOMER = "dim_customer"
DIM_DATE = "dim_date"

BRONZE_JOB = f"labrun-{LAB_SLUG}-bronze"
SILVER_JOB = f"labrun-{LAB_SLUG}-silver"
GOLD_JOB = f"labrun-{LAB_SLUG}-gold"
GLUE_ROLE = f"labrun-{LAB_SLUG}-role"
ATHENA_WG = f"labrun-{LAB_SLUG}-wg"

ACCOUNT_ID = None


In [ ]:
# NBVAL_SKIP
# Setup.

print("Paste your lab session token:")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (Enter to skip): ").strip() or None
AWS_CREDENTIALS = {"aws_access_key_id": AWS_ACCESS_KEY_ID, "aws_secret_access_key": AWS_SECRET_ACCESS_KEY, "aws_session_token": AWS_SESSION_TOKEN}

sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + orphan scan (multi-session resume-or-fresh).

CLEANUP_MANIFEST = [
    CleanupResource(type="glue_job", id=GOLD_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {GOLD_JOB} --region {REGION}"),
    CleanupResource(type="glue_job", id=SILVER_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {SILVER_JOB} --region {REGION}"),
    CleanupResource(type="glue_job", id=BRONZE_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {BRONZE_JOB} --region {REGION}"),
    CleanupResource(type="athena_workgroup", id=ATHENA_WG, region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {ATHENA_WG} --recursive-delete-option --region {REGION}"),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{DIM_DATE}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{DIM_CUSTOMER}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{DIM_PRODUCT}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{FACT_TABLE}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{SILVER_TABLE}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{BRONZE_TABLE}", region=REGION),
    CleanupResource(type="iam_role", id=GLUE_ROLE, region=REGION),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] Glue ETL bills only while running, but make sure cleanup ran.")


atexit.register(_atexit_cleanup)

tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_arns = [r["ResourceARN"] for r in tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]).get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed: {exc}")

if orphan_arns:
    print()
    print("Found existing labrun-tagged resources from a previous session:")
    for arn in orphan_arns:
        print(f"  EXISTING: {arn}")
    print()
    print("This lab is multi-session. Type 'resume' to continue with existing resources or 'fresh' to clean and start over.")
    choice = input("Choice: ").strip().lower()
    if choice == "resume":
        print("Resuming.")
    elif choice == "fresh":
        raise SystemExit("Run the previous session's cleanup cell first.")
    else:
        raise SystemExit("Invalid choice.")
else:
    print("No orphans. Manifest declared.")


## Seeding the raw clickstream

The next cell writes the raw clickstream JSON to S3. Mix of valid records and intentionally malformed ones:
- ~200 valid records
- 5 records with missing fields
- 3 records with wrong types (e.g., amount as string)
- 2 records that are not valid JSON

The bronze layer must tolerate every one of these.


In [ ]:
# NBVAL_SKIP
# Seed helper.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
except ClientError as exc:
    if exc.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
        raise

try:
    glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
except glue.exceptions.AlreadyExistsException:
    pass
try:
    athena.create_work_group(
        Name=ATHENA_WG,
        Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except Exception:
    pass

import random
from datetime import datetime, timedelta
rnd = random.Random(42)
categories = ["electronics", "books", "apparel", "home", "grocery"]
products = [f"sku-{i:03d}" for i in range(50)]
product_category = {p: rnd.choice(categories) for p in products}
customers = [f"cust-{i:04d}" for i in range(100)]

seed_lines = []
now = datetime(2026, 5, 15)
for i in range(200):
    day = now - timedelta(days=rnd.randint(0, 29))
    rec = {
        "event_id": str(uuid.uuid4()),
        "event_timestamp": day.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "customer_id": rnd.choice(customers),
        "product_id": rnd.choice(products),
        "amount_cents": rnd.randint(100, 50000),
        "event_date": day.strftime("%Y-%m-%d"),
    }
    seed_lines.append(json.dumps(rec))

# Add 5 records with missing fields (no amount_cents)
for i in range(5):
    seed_lines.append(json.dumps({"event_id": str(uuid.uuid4()), "event_timestamp": "2026-05-10T12:00:00Z", "customer_id": "cust-0001", "product_id": "sku-001"}))

# Add 3 records with wrong types (amount_cents as string)
for i in range(3):
    seed_lines.append(json.dumps({"event_id": str(uuid.uuid4()), "event_timestamp": "2026-05-12T12:00:00Z", "customer_id": "cust-0002", "product_id": "sku-002", "amount_cents": "not_a_number", "event_date": "2026-05-12"}))

# Add 2 totally broken lines
seed_lines.append("{not even json}")
seed_lines.append("garbage line")

# Save the product->category mapping for the gold dim conformance
s3.put_object(Bucket=BUCKET_NAME, Key="seed/product_categories.json", Body=json.dumps(product_category).encode("utf-8"))

# Late arrival to inject in Task 2
LATE_ARRIVAL = {
    "event_id": str(uuid.uuid4()),
    "event_timestamp": (now - timedelta(days=3)).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "customer_id": "cust-late",
    "product_id": "sku-001",
    "amount_cents": 99999,
    "event_date": (now - timedelta(days=3)).strftime("%Y-%m-%d"),
}

s3.put_object(Bucket=BUCKET_NAME, Key="raw/clickstream/page-1.jsonl", Body="\n".join(seed_lines).encode("utf-8"))
print(f"Raw seed written: {len(seed_lines)} lines (200 valid, 5 missing-fields, 3 type-mismatch, 2 garbage)")


## Task 1: Build the bronze layer (schema-on-read, malformed-tolerant)

Bronze keeps everything. The Glue ETL job reads `s3://<bucket>/raw/clickstream/*.jsonl`, tags each line with a `parse_error_reason` column when it fails to deserialize cleanly, and writes the result to `clickstream_bronze`.

The opinionated call: malformed records are tagged, not dropped. The validator will fail you if bronze has fewer rows than the raw input (i.e., if you dropped the garbage instead of keeping it).


In [ ]:
# NBVAL_SKIP
# Task 1: bronze IAM role + Glue ETL job.

GLUE_TRUST = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
GLUE_INLINE = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": ["s3:*", "athena:*", "glue:*", "logs:*"], "Resource": "*"}]}

try:
    iam.create_role(RoleName=GLUE_ROLE, AssumeRolePolicyDocument=json.dumps(GLUE_TRUST), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
    iam.attach_role_policy(RoleName=GLUE_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole")
    iam.put_role_policy(RoleName=GLUE_ROLE, PolicyName="labrun-glue-inline", PolicyDocument=json.dumps(GLUE_INLINE))
except iam.exceptions.EntityAlreadyExistsException:
    pass
glue_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GLUE_ROLE}"
print("  IAM propagation, hold 15 seconds...")
time.sleep(15)


def run_athena(query, timeout=120):
    qid = athena.start_query_execution(QueryString=query, WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return qid
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {qid} {s}: {athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status'].get('StateChangeReason')}")
        time.sleep(2)
    raise TimeoutError()


# Create the Iceberg bronze table.
run_athena(
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{BRONZE_TABLE} ("
    f"  event_id string, event_timestamp string, customer_id string, product_id string, "
    f"  amount_cents string, event_date string, payload string, parse_error_reason string"
    f") LOCATION 's3://{BUCKET_NAME}/{BRONZE_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)

BRONZE_SCRIPT = '''
import sys, json
from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, lit

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET", "DATABASE_NAME", "BRONZE_TABLE"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

# Schema-on-read: every column is STRING in bronze; we never lose data.
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount_cents", StringType(), True),
    StructField("event_date", StringType(), True),
])

# PERMISSIVE mode keeps malformed lines with a _corrupt_record column.
df = (
    spark.read.option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "payload")
         .schema(schema.add("payload", StringType(), True))
         .json(f"s3://" + args["BUCKET"] + "/raw/clickstream/*.jsonl")
)

# Tag parse errors. payload IS NULL means clean parse; payload IS NOT NULL
# means the entire input line came through as a string and the parse failed.
df_out = df.withColumn(
    "parse_error_reason",
    (col("payload").isNotNull()).cast("string"),
).withColumn("parse_error_reason",
    # store the reason as a string only when present, else NULL
    (col("payload").isNotNull()).cast("string"),
)

# Replace boolean string with a human reason on parse failures.
from pyspark.sql.functions import when
df_out = df_out.withColumn(
    "parse_error_reason",
    when(col("payload").isNotNull(), lit("invalid_json")).otherwise(lit(None).cast("string"))
)

df_out.createOrReplaceTempView("bronze_delta")
spark.sql(
    "INSERT INTO glue_catalog." + args["DATABASE_NAME"] + "." + args["BRONZE_TABLE"]
    + " (event_id, event_timestamp, customer_id, product_id, amount_cents, event_date, payload, parse_error_reason) "
    + " SELECT event_id, event_timestamp, customer_id, product_id, amount_cents, event_date, payload, parse_error_reason FROM bronze_delta"
)
print("LABRUN_BRONZE_OK")
'''

s3.put_object(Bucket=BUCKET_NAME, Key="scripts/bronze.py", Body=BRONZE_SCRIPT.encode("utf-8"))

iceberg_conf = (
    "spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog "
    f"--conf spark.sql.catalog.glue_catalog.warehouse=s3://{BUCKET_NAME}/warehouse/ "
    "--conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog "
    "--conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO "
    "--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
)

# YOUR CODE: create + start the bronze Glue job. Use the conf above as
# the --conf default argument. Wait for SUCCEEDED before moving on (about
# 4-5 minutes).


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: bronze contains all input rows including malformed.

def validator_1(session):
    try:
        qid = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{BRONZE_TABLE}")
        total = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
        qid2 = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{BRONZE_TABLE} WHERE parse_error_reason IS NOT NULL")
        errored = int(athena.get_query_results(QueryExecutionId=qid2)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator query failed: {exc}")
    if total < 200:
        return CheckpointResult("fail", error_reason=f"Bronze has {total} rows; expected at least 200. Are you dropping malformed records?")
    if errored < 2:
        return CheckpointResult("fail", error_reason=f"Bronze has only {errored} rows with parse_error_reason. Garbage lines must be retained with a non-null reason.")
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

PySpark's `spark.read.json` has a `mode` option. PERMISSIVE keeps malformed lines under a corrupt-record column.

</details>

<details><summary>Hint 2 (direction)</summary>

Set `mode=PERMISSIVE` and `columnNameOfCorruptRecord=payload`. Garbage lines land with the raw text in `payload` and all other fields NULL. Add a derived `parse_error_reason` column that fires when `payload IS NOT NULL`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
glue.create_job(
    Name=BRONZE_JOB, Role=glue_role_arn,
    Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/bronze.py", "PythonVersion": "3"},
    DefaultArguments={"--BUCKET": BUCKET_NAME, "--DATABASE_NAME": DATABASE_NAME, "--BRONZE_TABLE": BRONZE_TABLE,
                      "--datalake-formats": "iceberg", "--conf": iceberg_conf,
                      "--TempDir": f"s3://{BUCKET_NAME}/temp/"},
    GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
rid = glue.start_job_run(JobName=BRONZE_JOB)["JobRunId"]
while True:
    state = glue.get_job_run(JobName=BRONZE_JOB, RunId=rid)["JobRun"]["JobRunState"]
    if state in ("SUCCEEDED", "FAILED", "STOPPED", "ERROR", "TIMEOUT"):
        break
    time.sleep(15)
```

</details>


**Wallet check.** First Glue ETL run at the 10-minute minimum: ~$0.15. S3 puts: free. Iceberg writes: free at this volume. Session total so far: ~15 cents.

## Task 2: Build the silver layer (typed, deduped, late-arrival tolerant, partitioned by event_date)

Silver is the canonical layer. Type the columns, dedupe on `(event_id)`, partition by `day(event_date)` (Iceberg's hidden partitioning), filter out the parse_error_reason rows. Then inject the late arrival and re-run silver to prove the partition write goes to the correct historical day.


In [ ]:
# NBVAL_SKIP
# Task 2: silver Glue job + late-arrival injection.

run_athena(
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{SILVER_TABLE} ("
    f"  event_id string, event_timestamp timestamp, customer_id string, product_id string, "
    f"  amount_cents bigint, event_date date"
    f") PARTITIONED BY (day(event_date)) "
    f"LOCATION 's3://{BUCKET_NAME}/{SILVER_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)

SILVER_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, to_timestamp, to_date

args = getResolvedOptions(sys.argv, ["JOB_NAME", "DATABASE_NAME", "BRONZE_TABLE", "SILVER_TABLE"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

bronze = spark.read.table("glue_catalog." + args["DATABASE_NAME"] + "." + args["BRONZE_TABLE"]).filter(col("parse_error_reason").isNull())

typed = (
    bronze
    .withColumn("event_timestamp", to_timestamp("event_timestamp"))
    .withColumn("event_date", to_date("event_date"))
    .withColumn("amount_cents", col("amount_cents").cast("bigint"))
    .filter(col("amount_cents").isNotNull())
    .filter(col("event_id").isNotNull())
    .select("event_id", "event_timestamp", "customer_id", "product_id", "amount_cents", "event_date")
    .dropDuplicates(["event_id"])
)

typed.createOrReplaceTempView("silver_delta")

# MERGE on event_id; INSERT new rows, UPDATE existing.
spark.sql(
    "MERGE INTO glue_catalog." + args["DATABASE_NAME"] + "." + args["SILVER_TABLE"] + " t "
    "USING silver_delta s ON t.event_id = s.event_id "
    "WHEN MATCHED THEN UPDATE SET event_timestamp = s.event_timestamp, customer_id = s.customer_id, "
    "                              product_id = s.product_id, amount_cents = s.amount_cents, event_date = s.event_date "
    "WHEN NOT MATCHED THEN INSERT * "
)
print("LABRUN_SILVER_OK")
'''

s3.put_object(Bucket=BUCKET_NAME, Key="scripts/silver.py", Body=SILVER_SCRIPT.encode("utf-8"))

# YOUR CODE: create + start the silver Glue job. Wait for SUCCEEDED.

# YOUR CODE: inject the late-arrival record (3 days ago) into raw, re-run
# the bronze + silver pipeline, then verify the late record landed in
# the correct historical partition via SELECT count(*) FROM silver WHERE
# event_date = current_date - interval '3' day.

print()
print("Late-arrival injection: record below should be written to raw/clickstream/late.jsonl, "
      "then bronze + silver re-run.")
print(json.dumps(LATE_ARRIVAL))


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: silver typed, deduped, late arrival in correct partition.

def validator_2(session):
    try:
        # Schema check: amount_cents must be bigint, event_timestamp must be timestamp.
        table = glue.get_table(DatabaseName=DATABASE_NAME, Name=SILVER_TABLE)["Table"]
        cols = {c["Name"]: c["Type"] for c in table["StorageDescriptor"]["Columns"]}
        if cols.get("amount_cents") != "bigint":
            return CheckpointResult("fail", error_reason=f"amount_cents is {cols.get('amount_cents')!r}; expected bigint")
        if cols.get("event_timestamp") not in ("timestamp", "timestamp(6)"):
            return CheckpointResult("fail", error_reason=f"event_timestamp is {cols.get('event_timestamp')!r}; expected timestamp")
        # Dedup check
        qid = run_athena(f"SELECT count(*) AS total, count(DISTINCT event_id) AS distinct_ids FROM {DATABASE_NAME}.{SILVER_TABLE}")
        row = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]
        total = int(row["Data"][0]["VarCharValue"])
        distinct = int(row["Data"][1]["VarCharValue"])
        if total != distinct:
            return CheckpointResult("fail", error_reason=f"Silver has {total} rows but {distinct} distinct event_ids; dedup is incomplete.")
        # Late arrival check (3 days ago)
        qid2 = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{SILVER_TABLE} WHERE event_date = date '2026-05-12'")
        late = int(athena.get_query_results(QueryExecutionId=qid2)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
        if late < 1:
            return CheckpointResult("fail", error_reason=f"Silver has 0 rows for event_date 2026-05-12; did the late arrival land in the correct partition?")
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator failed: {exc}")
    return CheckpointResult("pass")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

Cast string columns to their right types. Drop the parse_error_reason rows. Dedupe on event_id.

</details>

<details><summary>Hint 2 (direction)</summary>

`to_timestamp("event_timestamp")`, `to_date("event_date")`, `col("amount_cents").cast("bigint")`. Use MERGE INTO so the late-arrival injection updates the existing row in place rather than creating duplicates.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The silver script (provided as SILVER_SCRIPT) is complete. Deploy the job:

```python
glue.create_job(
    Name=SILVER_JOB, Role=glue_role_arn,
    Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/silver.py", "PythonVersion": "3"},
    DefaultArguments={"--DATABASE_NAME": DATABASE_NAME, "--BRONZE_TABLE": BRONZE_TABLE, "--SILVER_TABLE": SILVER_TABLE,
                      "--datalake-formats": "iceberg", "--conf": iceberg_conf, "--TempDir": f"s3://{BUCKET_NAME}/temp/"},
    GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
rid = glue.start_job_run(JobName=SILVER_JOB)["JobRunId"]
# ... poll ...

# Late arrival
s3.put_object(Bucket=BUCKET_NAME, Key="raw/clickstream/late.jsonl", Body=(json.dumps(LATE_ARRIVAL) + "\n").encode())
# Re-run bronze + silver
```

</details>


**Wallet check.** Second Glue ETL run (silver): ~$0.15. Late-arrival re-run of bronze + silver: another ~$0.30. Session total: ~60 cents.

## Task 3: Build the gold star schema

Gold has one fact table and three conformed dimensions. The fact carries foreign keys, never the descriptive attributes. The dimensions carry the descriptive attributes (product_category, customer_signup_date, date components).


In [ ]:
# NBVAL_SKIP
# Task 3: gold star schema.

for tbl in [DIM_PRODUCT, DIM_CUSTOMER, DIM_DATE, FACT_TABLE]:
    try:
        glue.delete_table(DatabaseName=DATABASE_NAME, Name=tbl)
    except glue.exceptions.EntityNotFoundException:
        pass

run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{DIM_PRODUCT} ("
    f"  product_id string, product_category string"
    f") LOCATION 's3://{BUCKET_NAME}/{DIM_PRODUCT}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG')"
)
run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{DIM_CUSTOMER} ("
    f"  customer_id string, customer_segment string"
    f") LOCATION 's3://{BUCKET_NAME}/{DIM_CUSTOMER}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG')"
)
run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{DIM_DATE} ("
    f"  event_date date, day_of_week int, is_weekend boolean"
    f") LOCATION 's3://{BUCKET_NAME}/{DIM_DATE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG')"
)
run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{FACT_TABLE} ("
    f"  event_id string, event_date date, customer_id string, product_id string, amount_cents bigint"
    f") LOCATION 's3://{BUCKET_NAME}/{FACT_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG')"
)

# Load the dim_product from the saved product->category mapping.
product_category = json.loads(s3.get_object(Bucket=BUCKET_NAME, Key="seed/product_categories.json")["Body"].read())
dim_product_values = ", ".join(f"('{p}', '{c}')" for p, c in product_category.items())

GOLD_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, dayofweek, lit, when

args = getResolvedOptions(sys.argv, ["JOB_NAME", "DATABASE_NAME", "SILVER_TABLE", "FACT_TABLE", "DIM_CUSTOMER", "DIM_DATE", "BUCKET"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

silver = spark.read.table("glue_catalog." + args["DATABASE_NAME"] + "." + args["SILVER_TABLE"])

# fact: one row per event, FK to all three dims
fact = silver.select("event_id", "event_date", "customer_id", "product_id", "amount_cents").distinct()
fact.createOrReplaceTempView("fact_delta")
spark.sql("INSERT OVERWRITE TABLE glue_catalog." + args["DATABASE_NAME"] + "." + args["FACT_TABLE"] +
          " SELECT * FROM fact_delta")

# dim_customer: distinct customer_ids with a synthetic segment
dim_c = silver.select("customer_id").distinct().withColumn(
    "customer_segment",
    when(col("customer_id").startswith("cust-late"), lit("late"))
    .otherwise(lit("standard"))
)
dim_c.createOrReplaceTempView("dim_c_delta")
spark.sql("INSERT OVERWRITE TABLE glue_catalog." + args["DATABASE_NAME"] + "." + args["DIM_CUSTOMER"] +
          " SELECT customer_id, customer_segment FROM dim_c_delta")

# dim_date: distinct event_dates with day-of-week + weekend flag
dim_d = silver.select("event_date").distinct().withColumn(
    "day_of_week", dayofweek("event_date")
).withColumn("is_weekend", col("day_of_week").isin(1, 7))
dim_d.createOrReplaceTempView("dim_d_delta")
spark.sql("INSERT OVERWRITE TABLE glue_catalog." + args["DATABASE_NAME"] + "." + args["DIM_DATE"] +
          " SELECT * FROM dim_d_delta")

print("LABRUN_GOLD_OK")
'''

s3.put_object(Bucket=BUCKET_NAME, Key="scripts/gold.py", Body=GOLD_SCRIPT.encode("utf-8"))

# Load dim_product directly via Athena (it is small).
run_athena(f"INSERT INTO {DATABASE_NAME}.{DIM_PRODUCT} VALUES {dim_product_values}")

# YOUR CODE: create the gold Glue job + start it + wait for SUCCEEDED.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: gold star schema with referential integrity.

def validator_3(session):
    try:
        # FK from fact to dim_product
        qid = run_athena(
            f"SELECT count(*) FROM {DATABASE_NAME}.{FACT_TABLE} f "
            f"LEFT JOIN {DATABASE_NAME}.{DIM_PRODUCT} p ON f.product_id = p.product_id "
            f"WHERE p.product_id IS NULL"
        )
        orphan_products = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
        if orphan_products > 0:
            return CheckpointResult("fail", error_reason=f"{orphan_products} fact rows reference product_ids not in dim_product.")
        # FK from fact to dim_customer
        qid = run_athena(
            f"SELECT count(*) FROM {DATABASE_NAME}.{FACT_TABLE} f "
            f"LEFT JOIN {DATABASE_NAME}.{DIM_CUSTOMER} c ON f.customer_id = c.customer_id "
            f"WHERE c.customer_id IS NULL"
        )
        if int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]) > 0:
            return CheckpointResult("fail", error_reason="Fact rows reference customer_ids not in dim_customer.")
        # FK from fact to dim_date
        qid = run_athena(
            f"SELECT count(*) FROM {DATABASE_NAME}.{FACT_TABLE} f "
            f"LEFT JOIN {DATABASE_NAME}.{DIM_DATE} d ON f.event_date = d.event_date "
            f"WHERE d.event_date IS NULL"
        )
        if int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]) > 0:
            return CheckpointResult("fail", error_reason="Fact rows reference event_dates not in dim_date.")
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator failed: {exc}")
    return CheckpointResult("pass")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

Star schema: one fact, dims hold the descriptive attributes. The fact's columns are FKs (product_id, customer_id, event_date) plus the measure (amount_cents).

</details>

<details><summary>Hint 2 (direction)</summary>

Conform the dims FROM silver, not from raw. The validator checks anti-join referential integrity: every product_id in fact must appear in dim_product, etc.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
glue.create_job(
    Name=GOLD_JOB, Role=glue_role_arn,
    Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/gold.py", "PythonVersion": "3"},
    DefaultArguments={"--DATABASE_NAME": DATABASE_NAME, "--SILVER_TABLE": SILVER_TABLE, "--FACT_TABLE": FACT_TABLE,
                      "--DIM_CUSTOMER": DIM_CUSTOMER, "--DIM_DATE": DIM_DATE, "--BUCKET": BUCKET_NAME,
                      "--datalake-formats": "iceberg", "--conf": iceberg_conf, "--TempDir": f"s3://{BUCKET_NAME}/temp/"},
    GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
rid = glue.start_job_run(JobName=GOLD_JOB)["JobRunId"]
# poll for SUCCEEDED
```

</details>


**Wallet check.** Third Glue ETL run: ~$0.15. Athena scans for the FK checks: pennies. Session total so far: ~75 cents.

## Task 4: Analyst BI query returns the expected number; ground-truth from silver matches

The analyst ticket: "Show me revenue by product_category over the last 30 days." Run it against gold. Run the equivalent query against silver. Diff.


In [ ]:
# NBVAL_SKIP
# Task 4: BI query + ground-truth + diff.

GOLD_QUERY = f'''
SELECT p.product_category, sum(f.amount_cents) AS revenue_cents, count(*) AS event_count
FROM {DATABASE_NAME}.{FACT_TABLE} f
JOIN {DATABASE_NAME}.{DIM_PRODUCT} p ON f.product_id = p.product_id
JOIN {DATABASE_NAME}.{DIM_DATE} d ON f.event_date = d.event_date
WHERE f.event_date >= current_date - interval '30' day
GROUP BY p.product_category
ORDER BY p.product_category
'''

SILVER_QUERY = f'''
SELECT
  CASE WHEN s.product_id LIKE 'sku-001' OR s.product_id LIKE 'sku-002' THEN 'placeholder' ELSE 'placeholder' END AS marker,
  s.product_id, sum(s.amount_cents) AS revenue_cents, count(*) AS event_count
FROM {DATABASE_NAME}.{SILVER_TABLE} s
WHERE s.event_date >= current_date - interval '30' day
GROUP BY s.product_id
'''
# The silver query is intentionally lower-grain; we compute the ground-
# truth by joining to dim_product manually.

# YOUR CODE: run GOLD_QUERY; collect (product_category, revenue_cents, event_count) tuples.
# YOUR CODE: run a silver-side ground-truth query that joins silver to the
#   product->category mapping and computes the same aggregate.
# YOUR CODE: diff the two; assert empty diff.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: BI query matches ground-truth.

def validator_4(session):
    try:
        # Gold-side
        qid = run_athena(GOLD_QUERY)
        gold_rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1:]
        gold = {r["Data"][0]["VarCharValue"]: (int(r["Data"][1]["VarCharValue"]), int(r["Data"][2]["VarCharValue"])) for r in gold_rows}
        # Silver-side ground-truth via subquery + product->category mapping in dim_product
        gt_query = f'''
        SELECT p.product_category, sum(s.amount_cents) AS revenue_cents, count(*) AS event_count
        FROM {DATABASE_NAME}.{SILVER_TABLE} s
        JOIN {DATABASE_NAME}.{DIM_PRODUCT} p ON s.product_id = p.product_id
        WHERE s.event_date >= current_date - interval '30' day
        GROUP BY p.product_category
        ORDER BY p.product_category
        '''
        qid2 = run_athena(gt_query)
        gt_rows = athena.get_query_results(QueryExecutionId=qid2)["ResultSet"]["Rows"][1:]
        gt = {r["Data"][0]["VarCharValue"]: (int(r["Data"][1]["VarCharValue"]), int(r["Data"][2]["VarCharValue"])) for r in gt_rows}
        if gold != gt:
            diff = {k: (gold.get(k), gt.get(k)) for k in (set(gold) | set(gt)) if gold.get(k) != gt.get(k)}
            return CheckpointResult("fail", error_reason=f"Gold vs silver mismatch on {len(diff)} category groups: {dict(list(diff.items())[:3])}")
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator failed: {exc}")
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

Run the gold query as-is. Run a silver query that joins to dim_product for the category. The two answers must be equal.

</details>

<details><summary>Hint 2 (direction)</summary>

Just run the GOLD_QUERY string; the validator does the comparison. If you skipped Task 3's dim_product load, the join returns nothing and gold returns an empty result.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
qid = run_athena(GOLD_QUERY)
rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
for r in rows:
    print(r["Data"])
```

</details>


**Wallet check.** Two Athena scans for the BI + ground-truth: a fraction of a cent. Session total so far: ~75 cents.

## Task 5: Iceberg snapshot history shows the expected number of commits

Iceberg writes create snapshots. Three layers means three writes for gold's downstream tables (fact, dim_customer, dim_date). The lab tolerates 1-3 snapshots per gold table (the gold job uses INSERT OVERWRITE which creates one snapshot per run).


In [ ]:
# NBVAL_SKIP
# Task 5: SHOW HISTORY FOR <gold tables>; assert reasonable snapshot count.

# YOUR CODE: run SHOW HISTORY FOR <db>.fact_clickstream and assert
# snapshots between 1 and 5. Store the count in FACT_SNAPSHOT_COUNT.


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: snapshot count is in the expected range.

def validator_5(session):
    n = globals().get("FACT_SNAPSHOT_COUNT")
    if n is None:
        return CheckpointResult("fail", error_reason="FACT_SNAPSHOT_COUNT not set.")
    if not (1 <= n <= 5):
        return CheckpointResult("fail", error_reason=f"fact_clickstream has {n} snapshots; expected 1-5. If you have more, you wrote gold in many micro-batches instead of one batched run.")
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

`SHOW HISTORY FOR <table>` (or query the `<table>$snapshots` metadata table) returns one row per commit.

</details>

<details><summary>Hint 2 (direction)</summary>

In Athena (via Iceberg syntax) the metadata table is `"<table>$snapshots"`. Query that, count rows.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
qid = run_athena(f'SELECT count(*) FROM "{DATABASE_NAME}"."{FACT_TABLE}$snapshots"')
FACT_SNAPSHOT_COUNT = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
```

</details>


**Wallet check.** Tiny Athena scan against an Iceberg metadata table: nearly free. Session total: ~75 cents.

## Task 6: Portfolio artifact. Write the system-design markdown.

The writeup is the interview portfolio piece. Required H2 sections: Domain overview, Architecture diagram (you can render ASCII), Layer-by-layer design decisions (with subsections for bronze, silver, gold), Cost analysis, Future-state extensions. Word count: 800-1500.

Upload to `s3://<bucket>/portfolio/system-design.md`. The cleanup cell preserves it.


In [ ]:
# NBVAL_SKIP
# Task 6: write portfolio/system-design.md.

SYSTEM_DESIGN_TEMPLATE = '''# Medallion for the clickstream domain

## Domain overview

The clickstream domain captures customer purchase events: who bought what, how much, when. Upstream is a producer that emits semi-structured JSON to S3. Downstream is the analytics team that asks BI questions about revenue by category, by customer segment, by time. The medallion separates the dirty upstream from the clean analytical surface so neither side has to compromise.

Volume at lab scale is 200 events; at production scale this domain would handle 10M+ events per day. The architecture scales to that volume without structural changes; only sizing (worker counts on Glue, partition pruning on silver/gold) changes.

## Architecture diagram

```
[Producer] --> S3 raw/clickstream/*.jsonl
                  |
            [Glue ETL: bronze]   (schema-on-read, tolerant)
                  |
                  v
            clickstream_bronze   (Iceberg, every input row retained)
                  |
            [Glue ETL: silver]   (typed, deduped, partitioned by day(event_date))
                  |
                  v
            clickstream_silver   (Iceberg, canonical clean table)
                  |
            [Glue ETL: gold]     (star schema)
                  |
        +---------+---------+
        |         |         |
      fact   dim_product dim_customer dim_date
        \___ ___/
            v
         Athena analyst queries
```

## Layer-by-layer design decisions

### Bronze

Bronze keeps everything including malformed records. The opinionated call: schema-on-read, every column STRING, parse failures tagged with `parse_error_reason`. The trade-off is bronze storage size (we keep garbage forever); the upside is full lineage when the upstream contract breaks and we need to reconstruct what arrived.

### Silver

Silver is the canonical layer. Strong typing (timestamps, bigints, dates). Deduplication on event_id via MERGE. Hidden partitioning by `day(event_date)` so query pruning works without the analyst writing partition filters. Late arrivals (event_timestamp from days ago) land in their correct historical partition because MERGE writes to the right file regardless of when the row was processed.

### Gold

Gold is the star schema. One fact (`fact_clickstream`) and three conformed dims (`dim_product`, `dim_customer`, `dim_date`). The fact carries only FKs and the measure (amount_cents); dims carry descriptive attributes (product_category, customer_segment, day_of_week + is_weekend). Conformed dims are written FROM silver, not from raw, so they inherit silver's deduplication.

The fact is rebuilt with INSERT OVERWRITE so the snapshot history stays clean (one snapshot per gold run, not 50 micro-batches).

## Cost analysis

At lab volume: ~$2.50 per clean run, dominated by three Glue ETL runs at the 10-minute minimum each (2 DPU, ~$0.44 per DPU-hour = ~$0.15 per run). At production volume (10M events/day):

- Bronze: Glue Spark scan of the day's S3 prefix; ~$2.50 per day at 50 DPU-minutes.
- Silver: MERGE on the same volume; ~$3.00 per day.
- Gold: INSERT OVERWRITE rebuilds the fact; ~$5.00 per day.

Daily total ~$10 plus storage. The bigger production lever is partition pruning on the analyst's queries (silver query scans the last 30 days of partitions, not the full table; Athena pricing makes this a meaningful cost line).

## Future-state extensions

1. **Slowly Changing Dimension (SCD type 2) on dim_product.** Product categories shift over time; preserve the historical category at the moment of sale rather than re-categorizing retroactively.
2. **Lakehouse-side incremental gold.** Switch gold from INSERT OVERWRITE to MERGE so only changed partitions rebuild. Cuts cost at 100M events/day.
3. **Streaming silver via Kinesis + Glue streaming.** Reduce the latency from upstream event to silver availability from "next batch" to "5 minutes".
4. **dbt for transformations.** The PySpark scripts become dbt models with versioning + tests; the Glue jobs become orchestration only.
5. **Iceberg snapshot expiration on a 7-day window.** Production-scale write rates produce snapshot growth that slows metadata reads; expire automatically.
'''

# YOUR CODE: upload SYSTEM_DESIGN_TEMPLATE (or your customized version) to portfolio/system-design.md.


In [ ]:
# NBVAL_SKIP
# Checkpoint 6: portfolio markdown exists, required sections, word count 800-1500.

REQUIRED_SECTIONS = ["Domain overview", "Architecture diagram", "Layer-by-layer design decisions", "Cost analysis", "Future-state extensions"]


def _section_body(markdown, title):
    pattern = re.compile(rf"^##\s+{re.escape(title)}\s*$", re.MULTILINE)
    match = pattern.search(markdown)
    if not match:
        return None
    start = match.end()
    next_h2 = re.search(r"^##\s+", markdown[start:], re.MULTILINE)
    end = start + next_h2.start() if next_h2 else len(markdown)
    return markdown[start:end].strip()


def validator_6(session):
    try:
        body = s3.get_object(Bucket=BUCKET_NAME, Key="portfolio/system-design.md")["Body"].read().decode("utf-8")
    except ClientError:
        return CheckpointResult("fail", error_reason="portfolio/system-design.md not found.")
    missing = [s for s in REQUIRED_SECTIONS if _section_body(body, s) is None]
    if missing:
        return CheckpointResult("fail", error_reason=f"Missing H2 sections: {missing}")
    words = len(re.findall(r"\b\w+\b", body))
    if not (800 <= words <= 1500):
        return CheckpointResult("fail", error_reason=f"Word count is {words}; required range 800-1500.")
    return CheckpointResult("pass")


check(6, validator_6)


<details><summary>Hint 1 (nudge)</summary>

Five required H2 sections. Word count 800-1500. Write the design decisions you actually made.

</details>

<details><summary>Hint 2 (direction)</summary>

Use SYSTEM_DESIGN_TEMPLATE as a starting point. Customize the wording so it reflects what you built. Upload via `s3.put_object`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_object(Bucket=BUCKET_NAME, Key="portfolio/system-design.md", Body=SYSTEM_DESIGN_TEMPLATE.encode("utf-8"))
```

</details>


**Wallet check.** Free. Session total: ~75 cents.

In [ ]:
# NBVAL_SKIP
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "athena_workgroup":
            client = boto3.client("athena", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except Exception:
                pass
            return
        if resource.type == "s3_bucket":
            # Preserve portfolio/ contents in stdout before deleting the bucket.
            s3c = boto3.client("s3", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            print(f"  preserving portfolio/ before bucket delete:")
            try:
                listing = s3c.list_objects_v2(Bucket=resource.id, Prefix="portfolio/")
                for o in listing.get("Contents", []):
                    body = s3c.get_object(Bucket=resource.id, Key=o["Key"])["Body"].read().decode("utf-8")
                    print(f"  PORTFOLIO ARTIFACT: {o['Key']} ({len(body)} chars)")
                    print(body)
            except ClientError as exc:
                print(f"  portfolio preservation skipped: {exc}")
            try:
                while True:
                    listing = s3c.list_objects_v2(Bucket=resource.id)
                    objs = listing.get("Contents", [])
                    if not objs:
                        break
                    s3c.delete_objects(Bucket=resource.id, Delete={"Objects": [{"Key": o["Key"]} for o in objs]})
                    if not listing.get("IsTruncated"):
                        break
                s3c.delete_bucket(Bucket=resource.id)
            except ClientError:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("iceberg_table", "athena_workgroup"):
            return False
        return super().describe_resource(credentials, resource)


result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $2.50 to $5.00.** Most expensive lab in the catalog. Glue ETL (six+ runs across bronze/silver/gold + late-arrival reruns + any debug rounds) is the line item. Your system-design markdown is preserved in print output above; copy it for your portfolio before closing the notebook.

## These are not graded. They are for you.

**1. Which architectural decision was hardest to commit to?** Bronze tolerance, silver dedup, or the star schema in gold? Walk through what you would do differently if you ran this domain at 10x the data volume.

**2. Pre-aggregation in gold.** Imagine the analytics team comes back and asks for revenue by product_category at the day grain (one row per (date, category)). Do you compute that in silver, in a new gold aggregate table, at query time against fact, or in a downstream BI tool? Defend the call.

## Exam-Style Practice

**Question 1.** Your medallion ships. The analytics team asks for revenue at the daily product_category level. Where do you compute that aggregation?

A) In silver, by adding revenue per category per day to the silver table.

B) In gold as a separate aggregate table that drillthrough joins back to the fact.

C) At query time against gold; the BI tool handles the aggregation.

D) In a downstream BI tool's caching layer (e.g., Looker PDT).

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Silver is the canonical fact-grain layer. Pre-aggregating in silver locks you into one specific aggregation; you lose the ability to ask new questions (revenue by hour, revenue by subcategory) without rebuilding silver.
- B) Right. Star-schema gold supports both pre-aggregated tables (for known BI questions) AND the fact table (for new questions). A separate aggregate at the (date, category) grain answers the BI question fast; drillthrough to fact answers everything else.
- C) Wrong. Query time against fact works at small scale but does not at 10M events/day; the analyst's query scans the whole 30-day window every time. Cost scales with read volume.
- D) Wrong. BI-tool caching ties you to one tool. If the team switches from Looker to dbt+Lightdash next year, the cached aggregate moves; if it lives in the warehouse, it stays.

</details>

**Question 2.** Your silver layer's MERGE INTO step is partitioned by `day(event_date)`. A late-arriving record with event_timestamp from 3 days ago lands in the bronze. After the silver MERGE runs, which is the most likely outcome for that record?

A) The record lands in today's partition because that is when the MERGE processed it.

B) The record lands in the partition for 3 days ago, because Iceberg's hidden partitioning honors the row's event_date.

C) The record is rejected by silver because it falls outside the silver job's read window.

D) The record is silently dropped because silver does not support out-of-order inserts.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. The MERGE writes to a partition that Iceberg derives from the row's `event_date` column, not the wall-clock when the MERGE ran. Hidden partitioning is the entire point.
- B) Right. Iceberg's hidden partitioning means the row lands in the partition that matches its `event_date`. Three-day-old data lands in the three-day-old partition. Analyst queries scoped to the past week see the late arrival in its correct historical bucket.
- C) Wrong. Silver does not declare a read window; it MERGEs the whole bronze. Even if it did, "outside the window" would result in skipping, not rejecting.
- D) Wrong. Iceberg + MERGE is designed for exactly this case. Out-of-order arrivals are first-class.

</details>
